In [2]:
import pandas as pd
import altair as alt

if not hasattr(alt.utils, 'use_signature_func'):
    def _use_signature_func(_signature_func):
        def _decorator(func):
            return func
        return _decorator

    alt.utils.use_signature_func = _use_signature_func
import os

from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")  # or "article"

### Figure 1

In [3]:
fig1_data = pd.read_excel('Data/Figure1.xls')

fig1_data = fig1_data.melt(id_vars='Year', value_vars=['GDPpercapita_growth_trend', 'TFP'], value_name='value', var_name='metric')
fig1_data

,Year,metric,value
0,1945-01-01,GDPpercapita_growth_trend,-4.430616
1,1946-01-01,GDPpercapita_growth_trend,-2.569599
2,1947-01-01,GDPpercapita_growth_trend,-0.782892
3,1948-01-01,GDPpercapita_growth_trend,0.861876
4,1949-01-01,GDPpercapita_growth_trend,2.115471
...,...,...,...
157,2021-01-01,TFP,0.340275
158,2022-01-01,TFP,0.175551
159,2023-01-01,TFP,NaN
160,2024-01-01,TFP,NaN


In [25]:
fig1_data.dtypes

Year      datetime64[ns]
metric            object
value            float64
dtype: object

In [26]:
govt_year.dtypes

year          datetime64[ns]
government            object
dtype: object

In [23]:
govt_year = pd.read_csv('Data/uk_government_by_year.csv')

govt_year['year'] = pd.to_datetime(govt_year['year'], format='%Y')

In [24]:
fig1_data.merge(govt_year, how='left', left_on='Year', right_on='year')

,Year,metric,value,year,government
0,1945-01-01,GDPpercapita_growth_trend,-4.430616,1945-01-01,Labour
1,1946-01-01,GDPpercapita_growth_trend,-2.569599,1946-01-01,Labour
2,1947-01-01,GDPpercapita_growth_trend,-0.782892,1947-01-01,Labour
3,1948-01-01,GDPpercapita_growth_trend,0.861876,1948-01-01,Labour
4,1949-01-01,GDPpercapita_growth_trend,2.115471,1949-01-01,Labour
...,...,...,...,...,...
157,2021-01-01,TFP,0.340275,2021-01-01,Conservative
158,2022-01-01,TFP,0.175551,2022-01-01,Conservative
159,2023-01-01,TFP,NaN,2023-01-01,Conservative
160,2024-01-01,TFP,NaN,2024-01-01,Labour


In [35]:
styles.eco_colours

{'pink': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4',
 'green': '#00a767',
 'mid-blue': '#0063af',
 'purple': '#5c267b',
 'dot': '#f4134d',
 'grey': '#676a86'}

In [174]:
plot_data = fig1_data.merge(
    govt_year,
    how='left',
    left_on='Year',
    right_on='year'
)

y_scale = alt.Scale(domain=[-5, 5])
bar_data = plot_data.drop_duplicates('Year').assign(bar_min=-5, bar_max=5)

government_periods = (
    bar_data.sort_values('Year')
    .assign(period=lambda data: data['government'].ne(data['government'].shift()).cumsum())
    .groupby(['period', 'government'], as_index=False)
    .agg(start_year=('Year', 'min'), end_year=('Year', 'max'))
    .assign(
        label_year=lambda data: data['start_year'] + (data['end_year'] - data['start_year']) / 2,
        bar_label_y=3.85,
        label=lambda data: data['government'].map({'Labour': 'Lab', 'Conservative': 'Con'})
    )
)

bars = alt.Chart(bar_data).mark_bar(size=6.5, opacity=0.3).encode(
    x=alt.X('Year:T', title=None),
    y=alt.Y('bar_min:Q', scale=y_scale, title='Annual change'),
    y2='bar_max:Q',
    color=alt.Color(
        'government:N',
        scale=alt.Scale(domain=['Conservative', 'Labour'], range=['#0087dc', '#d50000']),
        legend=None
    )
)

government_labels = alt.Chart(government_periods).mark_text(
    align='center',
    baseline='top',
    dx=0,
    dy=-35,
    fontSize=14,
    fontWeight='bold'
).encode(
    x='label_year:T',
    y=alt.Y('bar_label_y:Q', scale=y_scale),
    text='label:N',
    color=alt.Color(
        'government:N',
        scale=alt.Scale(domain=['Conservative', 'Labour'], range=['#005a9c', '#9b0000']),
        legend=None
    )
)

zero_line = alt.Chart(pd.DataFrame({'value': [0]})).mark_rule(
    color='#808080',
    opacity=0.5
).encode(y=alt.Y('value:Q', scale=y_scale))

lines = alt.Chart(plot_data).mark_line(strokeWidth=2.5).encode(
    x=alt.X('Year:T', axis=alt.Axis(title='Year', labelFontSize=14)),
    y=alt.Y(
        'value:Q',
        scale=y_scale,
        axis=alt.Axis(title='Annual change', labelExpr="datum.value + '%'", labelFontSize=14, titleFontSize=14)
    ),
    color=alt.Color(
        'metric:N',
        scale=alt.Scale(domain=['GDPpercapita_growth_trend', 'TFP'], range=['#0063af', '#e6224b']),
        legend=None
    ),
    strokeDash=alt.StrokeDash(
        'metric:N',
        scale=alt.Scale(
            domain=['GDPpercapita_growth_trend', 'TFP'],
            range=[[1, 0], [6, 4]]
        ),
        legend=None
    ),
    tooltip=[
        alt.Tooltip('Year:T', title='Year', format='%Y'),
        alt.Tooltip('metric:N', title='Series'),
        alt.Tooltip('value:Q', title='Annual change (%)', format='.2f')
    ]
)

labels = alt.Chart(plot_data).transform_filter(
    'datum.value !== null'
).transform_window(
    rank='rank(Year)',
    sort=[alt.SortField('Year', order='descending')],
    groupby=['metric']
).transform_filter(
    'datum.rank === 1'
).transform_calculate(
    label_year='datetime(2025, 0, 1)',
    metric_label="datum.metric === 'GDPpercapita_growth_trend' ? 'GDP per capita' : 'TFP'"
).mark_text(align='left', dx=6, fontSize=14, fontWeight='bold').encode(
    x='label_year:T',
    y=alt.Y('value:Q', scale=y_scale),
    text=alt.Text('metric_label:N', title=None),
    color=alt.Color(
        'metric:N',
        scale=alt.Scale(domain=['GDPpercapita_growth_trend', 'TFP'], range=['#222222', '#F5A623']),
        legend=None
    )
)

fig1_chart = (bars + government_labels + zero_line + lines + labels).properties(
    width=525,
    height=325
).configure_view(stroke=None)

fig1_chart

alt.LayerChart(...)

In [175]:
# Save to png
fig1_chart.save('fig1_chart.png', scale_factor=2)
# Save to json
fig1_chart.save('fig1_chart.json', scale_factor=2)

# Figure 2

In [75]:
fig2and3_data = pd.read_excel('Data/Fig2and3.xlsx')

In [83]:
fig2_data = fig2and3_data.iloc[:, :7]

In [85]:
fig2_data.columns

Index(['Country', 'Code', 'GDP_1950', 'Gr_1950-1972', 'GR_1950-1989',
       'Gr_1950-2006', 'Gr_1950-2022'],
      dtype='object')

In [112]:
styles.eco_colours

{'pink': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4',
 'green': '#00a767',
 'mid-blue': '#0063af',
 'purple': '#5c267b',
 'dot': '#f4134d',
 'grey': '#676a86'}

In [185]:
period_order = ['1950-1972', '1950-1989', '1950-2006', '1950-2022']
period_labels = {
    'Gr_1950-1972': '1950-1972',
    'GR_1950-1989': '1950-1989',
    'Gr_1950-2006': '1950-2006',
    'Gr_1950-2022': '1950-2022'
}

tooltip_fields = [
    alt.Tooltip('Country:N', title='Country'),
    alt.Tooltip('GDP_1950:Q', title='GDP per capita, 1950 (2011 prices)', format='$,.0f'),
    alt.Tooltip('growth:Q', title='Average annual growth', format='.2f')
]

fig2_long = fig2_data.melt(
    id_vars=['Country', 'Code', 'GDP_1950'], value_vars=list(period_labels),
    var_name='period', value_name='growth'
).assign(period=lambda data: data['period'].map(period_labels))

non_uk_points = alt.Chart(fig2_long).transform_filter(
    "datum.Country !== 'United Kingdom'"
).mark_circle(
    filled=False, color='#0063af', opacity=0.45, strokeWidth=1.25, size=55
).encode(
    x=alt.X('GDP_1950:Q', axis=alt.Axis(title='GDP per capita, 1950 (2011 prices)', titleFontSize=14, labelFontSize=14, labelExpr="'$' + datum.value / 1000 + 'k'")),
    y=alt.Y('growth:Q', axis=alt.Axis(title='GDP per capita growth rate', titleFontSize=14, labelFontSize=14, labelExpr="datum.value + '%'")),
    tooltip=tooltip_fields
)

uk_point = alt.Chart(fig2_long).transform_filter(
    "datum.Country === 'United Kingdom'"
).mark_circle(color='#e6224b', size=140).encode(
    x='GDP_1950:Q', y='growth:Q', tooltip=tooltip_fields
)

uk_label = alt.Chart(fig2_long).transform_filter(
    "datum.Country === 'United Kingdom'"
).mark_text(
    align='left', dx=7, fontWeight='bold', color='#e6224b', fontSize=13
).encode(x='GDP_1950:Q', y='growth:Q', text='Country:N')

trend = alt.Chart(fig2_long).transform_regression(
    'GDP_1950', 'growth'
).mark_line(
    color='#676A86', strokeWidth=1.5, strokeDash=[6, 4]
).encode(x='GDP_1950:Q', y='growth:Q')

fig2_chart = (trend + non_uk_points + uk_point + uk_label).properties(
    width=275, height=220
).facet(
    facet=alt.Facet(
        'period:N', title=None, sort=period_order,
        header=alt.Header(labelFontSize=16, labelColor='#676a86', labelPadding=20)
    ),
    columns=2,
    spacing=16
).resolve_scale(x='independent')

fig2_chart

alt.FacetChart(...)

In [189]:
# Save to png
fig2_chart.save('fig2_chart.png', scale_factor=2)
# Save to json
fig2_chart.save('fig2_chart.json', scale_factor=2)

# Figure 3

In [122]:
fig2and3_data.columns

Index(['Country', 'Code', 'GDP_1950', 'Gr_1950-1972', 'GR_1950-1989',
       'Gr_1950-2006', 'Gr_1950-2022', 'Unnamed: 7', 'TFPgr_1950-1972',
       'TFPgr_1950-1989', 'TFPgr_1950-2006', 'TFPgr_1950-2022'],
      dtype='object')

In [187]:
fig3_period_order = ['1950-1972', '1950-1989', '1950-2006', '1950-2022']
fig3_columns = {
    '1950-1972': ('Gr_1950-1972', 'TFPgr_1950-1972'),
    '1950-1989': ('GR_1950-1989', 'TFPgr_1950-1989'),
    '1950-2006': ('Gr_1950-2006', 'TFPgr_1950-2006'),
    '1950-2022': ('Gr_1950-2022', 'TFPgr_1950-2022')
}

fig3_long = pd.concat([
    fig2and3_data[['Country', 'Code', growth_column, tfp_column]]
    .rename(columns={growth_column: 'growth', tfp_column: 'tfp_growth'})
    .assign(period=period)
    for period, (growth_column, tfp_column) in fig3_columns.items()
], ignore_index=True).dropna(subset=['growth', 'tfp_growth'])

fig3_x_scale = alt.Scale(domain=[0, 5])
fig3_tooltips = [
    alt.Tooltip('Country:N', title='Country'),
    alt.Tooltip('tfp_growth:Q', title='Average annual TFP growth', format='.2f'),
    alt.Tooltip('growth:Q', title='GDP per capita growth rate', format='.2f')
]

fig3_non_uk_points = alt.Chart(fig3_long).transform_filter(
    "datum.Country !== 'United Kingdom'"
).mark_circle(
    filled=False, color='#0063af', opacity=0.45, strokeWidth=1.25, size=55
).encode(
    x=alt.X('tfp_growth:Q', scale=fig3_x_scale, axis=alt.Axis(title='Average annual TFP growth (%)', titleFontSize=14, labelFontSize=14, labelExpr="datum.value + '%'")),
    y=alt.Y('growth:Q', axis=alt.Axis(title='GDP per capita growth rate', titleFontSize=14, labelFontSize=14, labelExpr="datum.value + '%'")),
    tooltip=fig3_tooltips
)

fig3_uk_point = alt.Chart(fig3_long).transform_filter(
    "datum.Country === 'United Kingdom'"
).mark_circle(color='#e6224b', size=140).encode(
    x=alt.X('tfp_growth:Q', scale=fig3_x_scale), y='growth:Q', tooltip=fig3_tooltips
)

fig3_uk_label = alt.Chart(fig3_long).transform_filter(
    "datum.Country === 'United Kingdom'"
).mark_text(
    align='left', dx=7, fontWeight='bold', color='#e6224b', fontSize=13
).encode(x=alt.X('tfp_growth:Q', scale=fig3_x_scale), y='growth:Q', text='Country:N')

fig3_trend = alt.Chart(fig3_long).transform_regression(
    'tfp_growth', 'growth'
).mark_line(
    color='#676A86', strokeWidth=1.5, strokeDash=[6, 4]
).encode(x=alt.X('tfp_growth:Q', scale=fig3_x_scale), y='growth:Q')

fig3_chart = (fig3_trend + fig3_non_uk_points + fig3_uk_point + fig3_uk_label).properties(
    width=275, height=220
).facet(
    facet=alt.Facet(
        'period:N', title=None, sort=fig3_period_order,
        header=alt.Header(labelFontSize=16, labelColor='#676a86', labelPadding=20)
    ),
    columns=2,
    spacing=30
).resolve_scale(x='independent')

fig3_chart

alt.FacetChart(...)

In [188]:
# Save to png
fig3_chart.save('fig3_chart.png', scale_factor=2)
# Save to json
fig3_chart.save('fig3_chart.json', scale_factor=2)

# Figure 4

In [208]:
mpd_gdp = pd.read_excel('Data/mpd2023_web.xlsx', sheet_name='GDPpc', skiprows=2)
mpd_gdp['year'] = pd.to_numeric(mpd_gdp['year'])
mpd_gdp = mpd_gdp[mpd_gdp['year'] >= 1700]
mpd_gdp['year'] = pd.to_datetime(mpd_gdp['year'], format='%Y')
keep_countries = ['GBR', 'CAN', 'DEU', 'FRA', 'ITA', 'JPN', 'USA']
mpd_gdp = mpd_gdp[['year'] + keep_countries]

In [209]:
mpd_gdp = mpd_gdp.reset_index(drop=True)

In [211]:
mpd_gdp['GBR_index'] = mpd_gdp['GBR'] / mpd_gdp['GBR'] * 100
mpd_gdp['CAN_index'] = mpd_gdp['CAN'] / mpd_gdp['GBR'] * 100
mpd_gdp['DEU_index'] = mpd_gdp['DEU'] / mpd_gdp['GBR'] * 100
mpd_gdp['FRA_index'] = mpd_gdp['FRA'] / mpd_gdp['GBR'] * 100
mpd_gdp['ITA_index'] = mpd_gdp['ITA'] / mpd_gdp['GBR'] * 100
mpd_gdp['JPN_index'] = mpd_gdp['JPN'] / mpd_gdp['GBR'] * 100
mpd_gdp['USA_index'] = mpd_gdp['USA'] / mpd_gdp['GBR'] * 100

In [212]:
mpd_gdp_indexes = mpd_gdp[['year', 'GBR_index', 'CAN_index', 'DEU_index', 'FRA_index', 'ITA_index', 'JPN_index', 'USA_index']]

In [241]:
styles.eco_colours

{'pink': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4',
 'green': '#00a767',
 'mid-blue': '#0063af',
 'purple': '#5c267b',
 'dot': '#f4134d',
 'grey': '#676a86'}

In [250]:
import altair as alt
import pandas as pd

# --- 1. Reshape from wide to long ---------------------------------------
df = mpd_gdp_indexes.copy()
df['year'] = pd.to_datetime(df['year']).dt.year

name_map = {
    'GBR_index': 'UK',      'CAN_index': 'Canada',
    'DEU_index': 'Germany', 'FRA_index': 'France',
    'ITA_index': 'Italy',   'JPN_index': 'Japan',
    'USA_index': 'US',
}

long = (
    df.melt(id_vars='year', value_vars=list(name_map),
            var_name='country', value_name='gdp_index')
      .assign(country=lambda d: d['country'].map(name_map))
      .dropna(subset=['gdp_index'])
)

year_min, year_max = int(df['year'].min()), int(df['year'].max())

# --- 2. Era bands -------------------------------------------------------
eras = pd.DataFrame([
    {'start': year_min, 'end': 1870,     'label': 'British Dominance',
     'fill': '#f2c9c9', 'text_color': '#b23b3b'},
    {'start': 1870,     'end': 1945,     'label': 'Relative Decline',
     'fill': '#cbcbee', 'text_color': '#3f3fa0'},
    {'start': 1945,     'end': year_max, 'label': 'British Disease',
     'fill': '#dcc9ec', 'text_color': '#7b3fa0'},
]).assign(mid=lambda d: (d['start'] + d['end']) / 2)

# --- 3. Scales / colour order -------------------------------------------
order  = ['UK', 'Canada', 'Germany', 'France', 'Italy', 'Japan', 'US']
colors = ['#000000', '#e6224b', '#122b39', '#0063af', '#00a767', '#f4c245', '#179fdb']
dashes = [[6, 4]] + [[1, 0]] * 6

x_scale = alt.Scale(domain=[year_min, year_max + 6], nice=False)  # +6 pad for labels
y_scale = alt.Scale(domain=[15, 165])
color_scale = alt.Scale(domain=order, range=colors)

# --- 4. End-of-line labels ----------------------------------------------
# take each country's last available point, then hand-tune vertical offsets
end_pts = (long.sort_values('year')
               .groupby('country', as_index=False)
               .last())

# dy in pixels: negative = up, positive = down. Tweak to taste.
dy_offsets = {
    'US':      0,
    'Canada':   5,
    'Germany':  -5,
    'France':  -15,
    'Italy':    10,
    'Japan':   7,
    'UK':       -5,
}
end_pts['dy'] = end_pts['country'].map(dy_offsets).fillna(0)

# --- 5. Layers ----------------------------------------------------------
background = alt.Chart(eras).mark_rect(opacity=0.3).encode(
    x=alt.X('start:Q', scale=x_scale, title=None), x2='end:Q',
    color=alt.Color('fill:N', scale=None, legend=None),
)

era_labels = alt.Chart(eras).mark_text(
    baseline='top', dx=-50, dy=4, fontSize=14, fontWeight='bold'
).encode(
    x=alt.X('mid:Q', scale=x_scale), y=alt.value(6),
    text='label:N',
    color=alt.Color('text_color:N', scale=None, legend=None),
)

lines = alt.Chart(long).mark_line().encode(
    x=alt.X('year:Q', scale=x_scale,
            axis=alt.Axis(format='d', title=None, labelFontSize=14,
                          values=[1700, 1750, 1800, 1850, 1900, 1950, 2000])),
    y=alt.Y('gdp_index:Q', scale=y_scale, title='',
            axis=alt.Axis(values=[20, 60, 100, 140], titleFontSize=14, labelFontSize=14,)),
    color=alt.Color('country:N', scale=color_scale, legend=None),
    strokeDash=alt.StrokeDash('country:N',
                    scale=alt.Scale(domain=order, range=dashes), legend=None),
)

end_labels = alt.layer(*[
    alt.Chart(end_pts[end_pts.country == c]).mark_text(
        align='left', dx=4, dy=dy_offsets[c], fontSize=14, fontWeight='bold'
    ).encode(x='year:Q', y='gdp_index:Q', text='country:N',
             color=alt.Color('country:N', scale=color_scale, legend=None))
    for c in order
])

tooltips = [
    alt.Tooltip('country:N', title='Country'),
    alt.Tooltip('year:Q',    title='Year',            format='d'),
    alt.Tooltip('gdp_index:Q', title='GDP (UK=100)',  format='.1f'),
]

lines = alt.Chart(long).mark_line().encode(
    x=alt.X('year:Q', scale=x_scale,
            axis=alt.Axis(format='d', title=None, labelFontSize=14,
                          values=[1700, 1750, 1800, 1850, 1900, 1950, 2000])),
    y=alt.Y('gdp_index:Q', scale=y_scale, title='',
            axis=alt.Axis(values=[20, 60, 100, 140], titleFontSize=14, labelFontSize=14)),
    color=alt.Color('country:N', scale=color_scale, legend=None),
    strokeDash=alt.StrokeDash('country:N',
                    scale=alt.Scale(domain=order, range=dashes), legend=None),
    tooltip=tooltips,
)

# invisible points so the tooltip catches anywhere, not just on vertices
hover = alt.Chart(long).mark_circle(size=30, opacity=0).encode(
    x=alt.X('year:Q', scale=x_scale),
    y=alt.Y('gdp_index:Q', scale=y_scale),
    color=alt.Color('country:N', scale=color_scale, legend=None),
    tooltip=tooltips,
)

fig4_chart = (
    alt.layer(background, era_labels, lines, hover, end_labels)
       .resolve_scale(color='independent', strokeDash='independent')
       .properties(width=560, height=340)
       .configure_view(strokeWidth=0)
       .configure_axis(grid=True, gridColor='#eeeeee', domainColor='#cccccc')
)

fig4_chart

alt.LayerChart(...)

In [251]:
# Save to png
fig4_chart.save('fig4_chart.png', scale_factor=2)
# Save to json
fig4_chart.save('fig4_chart.json', scale_factor=2)

In [252]:
styles.eco_colours

{'pink': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4',
 'green': '#00a767',
 'mid-blue': '#0063af',
 'purple': '#5c267b',
 'dot': '#f4134d',
 'grey': '#676a86'}